### Import libraries

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import StandardScaler

### Load game and extra-statistics data

In [2]:
data_dir = Path("data") if Path("data").is_dir() else Path("../data")

ids = {"game_id": "string", "season_id": "string", "team_id_home": "string", "team_id_away": "string"}
games = pd.read_csv(data_dir / "game.csv", dtype=ids, parse_dates=["game_date"])
# other_stats has 10 exact duplicate rows (all outside the selected seasons)
stats = pd.read_csv(data_dir / "other_stats.csv", dtype=ids).drop_duplicates()

### Keep regular-season games from 2010–11 onward that have extra statistics

In [3]:
# all_games becomes an unfiltered copy used for feature engineering later.
all_games = games.sort_values(["game_date", "game_id"])

season_year = games["season_id"].str[-4:].astype(int)
season_games = games[(games["season_type"] == "Regular Season") & (season_year >= 2010)]
games = season_games[season_games["game_id"].isin(stats["game_id"])]
print("Games:", len(games))

Games: 12074


### One row per team per game

In [4]:
def stack_sides(df, cols, keep=("game_id",)):
    """Turn *_home / *_away column pairs into one row per team."""
    keep = list(keep)
    return pd.concat(
        df[keep + [f"{c}_{side}" for c in cols]].set_axis(keep + cols, axis=1).assign(side=side)
        for side in ("home", "away")
    )


box = stack_sides(
    games,
    ["team_id", "wl", "pts", "fgm", "fga", "fg3m", "fg3a", "reb", "tov"],
    keep=("game_id", "game_date", "season_id"),
)
extra = stack_sides(stats, ["team_id", "pts_paint", "pts_2nd_chance", "pts_fb", "pts_off_to"]).drop(columns="side")

# other_stats has home/away swapped in about half its games, so match on team_id, not side.
team_games = box.merge(extra, on=["game_id", "team_id"], validate="one_to_one")
team_games["win"] = (team_games["wl"] == "W").astype(float)
team_games["pts_allowed"] = team_games.groupby("game_id")["pts"].transform("sum") - team_games["pts"]

### Five-game rolling means (previous games only, same team and season)

In [5]:
WINDOW = 5
features = {
    "win_rate": "win",
    "points_scored_mean": "pts",
    "points_allowed_mean": "pts_allowed",
    "rebounds_mean": "reb",
    "turnovers_mean": "tov",
    "paint_points_mean": "pts_paint",
    "second_chance_points_mean": "pts_2nd_chance",
    "fast_break_points_mean": "pts_fb",
    "points_off_turnovers_mean": "pts_off_to",
}
source_cols = list(features.values()) + ["fgm", "fga", "fg3m", "fg3a"]

team_games = team_games.sort_values(["game_date", "game_id"])
rolled = team_games.groupby(["team_id", "season_id"])[source_cols].transform(
    # shift(1) first so the current game never feeds its own predictors
    lambda s: s.shift(1).rolling(WINDOW).mean()
)

history = team_games[["game_id", "side"]].copy()
for name, col in features.items():
    history[name] = rolled[col]
# Ratio of 5-game means equals ratio of 5-game totals
history["fg_pct"] = rolled["fgm"] / rolled["fga"]
history["fg3_pct"] = rolled["fg3m"] / rolled["fg3a"]

### Elo ratings (all regular-season and playoff games since 1983–84)

A margin-based Elo rating: each team's rating is its strength **in points relative to an average team**, built from every game since 1983–84.

- Before each gamethe expected home margin is home rating + 3 (home court advantage) − away rating.
- After each game both ratings move by k × (actual margin − expected margin). 

With `k = 0.03`, a game's influence halves after around 23 of a team's games, so the rating reflects mostly the current season plus some of last season. `elo_diff` is therefore a predicted home point differential.


In [ ]:
def elo_ratings(games, k=0.03, home_adv=3, carry=0.65, cap=20):
    """Pre-game team strength, in points relative to an average team."""
    rating, last_season, rows = {}, {}, []
    for g in games.itertuples():
        season = g.season_id[-4:]
        for team in (g.team_id_home, g.team_id_away):
            if last_season.get(team) != season:
                rating[team] = carry * rating.get(team, 0.0)  
            last_season[team] = season

        home, away = rating[g.team_id_home], rating[g.team_id_away]
        rows.append((g.game_id, home, away))

        expected = home + home_adv - away
        actual = min(max(g.pts_home - g.pts_away, -cap), cap)
        change = k * (actual - expected)
        rating[g.team_id_home] += change
        rating[g.team_id_away] -= change

    return pd.DataFrame(rows, columns=["game_id", "elo_home", "elo_away"]).set_index("game_id")


elo_games = all_games[
    all_games["season_type"].isin(["Regular Season", "Playoffs"]) & (all_games["game_date"] >= "1983-10-01")
]
elo = elo_ratings(elo_games)
elo["elo_diff"] = elo["elo_home"] - elo["elo_away"]

### Net rating and rest days (previous games only, same team and season)

Net rating is point margin per 100 possessions, so it adjusts for pace. We estimate it as `FGA - OREB + TOV + 0.44 * FTA`, averaged over both teams. These use all 2010 and ownwards regular-season games. 

- `net_rating_last10`: last 10 games
- `net_rating_season`: season to date
- `rest_days`: days since the team's previous game (capped at 7)
- `back_to_back`: 1 if the team played yesterday

In [7]:
net = stack_sides(
    season_games,
    ["team_id", "pts", "fga", "fta", "oreb", "tov"],
    keep=("game_id", "game_date", "season_id"),
).sort_values(["game_date", "game_id"])

net["poss"] = net["fga"] - net["oreb"] + net["tov"] + 0.44 * net["fta"]
net["poss"] = net.groupby("game_id")["poss"].transform("mean")
net["margin"] = 2 * net["pts"] - net.groupby("game_id")["pts"].transform("sum")

by_team = net.groupby(["team_id", "season_id"])[["margin", "poss"]]
last10 = by_team.transform(lambda s: s.shift(1).rolling(10, min_periods=1).sum())
to_date = by_team.transform(lambda s: s.shift(1).expanding().sum())

net["net_rating_last10"] = 100 * last10["margin"] / last10["poss"]
net["net_rating_season"] = 100 * to_date["margin"] / to_date["poss"]
net["rest_days"] = net.groupby("team_id")["game_date"].diff().dt.days.clip(upper=7).fillna(7)
net["back_to_back"] = (net["rest_days"] == 1).astype(int)

extra_stems = ["net_rating_last10", "net_rating_season", "rest_days", "back_to_back"]
extra_wide = net.pivot(index="game_id", columns="side", values=extra_stems)
extra_wide.columns = [f"{side}_{stem}" for stem, side in extra_wide.columns]

extra_cols = ["elo_home", "elo_away", "elo_diff"] + [f"{side}_{stem}" for side in ("home", "away") for stem in extra_stems]

### One row per game; require five prior games for both teams

In [8]:
stems = [*features, "fg_pct", "fg3_pct"]
feature_cols = [f"{side}_{stem}_last{WINDOW}" for side in ("home", "away") for stem in stems]

wide = history.pivot(index="game_id", columns="side").dropna()
wide.columns = [f"{side}_{stem}_last{WINDOW}" for stem, side in wide.columns]

### Add labels and build the final dataset

In [9]:
games = games.set_index("game_id")
labels = pd.DataFrame({
    "home_win": (games["wl_home"] == "W").astype(int),
    "point_differential": games["pts_home"] - games["pts_away"],
    "pts_home": games["pts_home"],
    "pts_away": games["pts_away"],
})

df = (
    games[["game_date", "season_id", "team_id_home", "team_id_away"]]
    .join(wide[feature_cols], how="inner")
    .join(elo)
    .join(extra_wide)
    .join(labels)
    .sort_values(["game_date", "game_id"])
)
assert len(df) == 11_086, len(df)

### Scale features



In [10]:
df[feature_cols + extra_cols] = StandardScaler().fit_transform(df[feature_cols + extra_cols])

print(df.shape)
df.head()

(11086, 41)


,game_date,season_id,team_id_home,team_id_away,home_win_rate_last5,home_points_scored_mean_last5,home_points_allowed_mean_last5,home_rebounds_mean_last5,home_turnovers_mean_last5,home_paint_points_mean_last5,...,away_net_rating_season,home_net_rating_season,away_rest_days,home_rest_days,away_back_to_back,home_back_to_back,home_win,point_differential,pts_home,pts_away
game_id,,,,,,,,,,,,,,,,,,,,,
0021000074,2010-11-05,22010,1610612750,1610612737,-1.147144,-1.032638,0.882958,1.559222,2.775485,-1.070859,...,1.793784,-3.160471,0.014973,-0.185368,-0.601456,-0.417123,0,-10.0,103.0,113.0
0021000083,2010-11-06,22010,1610612749,1610612740,-0.378313,-1.694900,-1.380728,-0.632014,1.193575,-1.694033,...,1.240201,-0.621045,-1.069922,-1.281949,1.662632,2.397376,0,-6.0,81.0,87.0
0021000088,2010-11-06,22010,1610612758,1610612763,0.390519,0.132028,0.379917,-0.519643,-1.179288,-0.011462,...,-0.757458,-0.443077,-1.069922,0.911212,1.662632,-0.417123,0,-9.0,91.0,100.0
0021000089,2010-11-07,22010,1610612752,1610612755,0.390519,-0.073501,-0.649031,-0.013973,0.798098,-0.073779,...,-0.275531,0.978535,0.014973,-0.185368,-0.601456,-0.417123,0,-10.0,96.0,106.0
0021000090,2010-11-07,22010,1610612737,1610612756,1.928182,-0.438887,-1.312131,-0.407272,0.204882,-0.634636,...,-0.125443,1.824462,0.014973,-0.185368,-0.601456,-0.417123,0,-4.0,114.0,118.0


In [11]:
# Validate and save the prepared dataset
assert df.index.is_unique
assert df[feature_cols + extra_cols].notna().all().all()
assert df[["home_win", "point_differential", "pts_home", "pts_away"]].notna().all().all()
assert set(df["home_win"].unique()) == {0, 1}

processed_path = data_dir / "processed_games.csv"
df.to_csv(processed_path)
print(f"Saved {len(df):,} games with {len(feature_cols) + len(extra_cols)} scaled features to {processed_path}")

Saved 11,086 games with 33 scaled features to ..\data\processed_games.csv
